<a href="https://colab.research.google.com/github/aarsh121/Financial-Analysis-using-Python/blob/main/Prediction_Bitcoin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importing Libraries**

In [ ]:
import os
import pandas as pd
import yfinance as yf
from datetime import datetime
import time

In [ ]:
# Using yahoo finance to fetch the bitcoin data

btc_ticker=yf.Ticker('BTC-USD')
btc=btc_ticker.history(period='max')

In [ ]:
btc

In [ ]:
#Deleting the unnecessary columns
btc.drop(['Dividends', 'Stock Splits'],axis=1, inplace=True)

In [ ]:
btc.index=pd.to_datetime(btc.index)

In [ ]:
btc.index = btc.index.date

In [ ]:
btc

In [ ]:
#changing column name to lower case
btc.columns=[c.lower() for c in btc.columns]

In [ ]:
btc.plot.line(y='close', use_index=True)

As we can see we should have buy the Bitcoin in 2020, after that it exploded. BIG LOSS who didn't buy
.

In [ ]:
# Reading wiki sentiment file

wiki=pd.read_csv('/content/wikipedia_edits.csv', index_col=0, parse_dates=True)
wiki

In [ ]:
#Merge both data set on the index column
btc=btc.merge(wiki, left_index=True, right_index=True)

In [ ]:
btc

In [ ]:
#create new column - tommorow which will have the next day closing value
btc['tommorow']=btc['close'].shift(-1)

In [ ]:
btc

In [ ]:
#define target column if tmrw>close then 1 else 0.
btc['target']=(btc['tommorow']> btc['close']).astype(int)


In [ ]:
# we can see below that our data is symmetric, A balanced data set
btc['target'].value_counts()

In [ ]:
#Uisng Random Forest Classifier as our model

from sklearn.ensemble import RandomForestClassifier

model=RandomForestClassifier(n_estimators=100, min_samples_split=50, random_state=1)

train=btc.iloc[:-200]
test=btc.iloc[-200:]

predictors=['close','open', 'volume', 'high', 'low', 'edit_count', 'neg_sentiment', 'sentiment']
model.fit(train[predictors], train['target'])


In [ ]:
from sklearn.metrics import precision_score

preds=model.predict(test[predictors])
preds= pd.Series(preds, index=test.index)
precision_score(test['target'], preds)


**We can see that our base line model is giving the accuracy of 64%. Lets see if hyper parameter tuning can increase the precision score.**

In [ ]:
def predict(train, test, predictors, model):
  model.fit(train[predictors], train['target'])
  preds=model.predict(test[predictors])
  preds= pd.Series(preds, index=test.index, name='predictions')
  combined =pd.concat([test['target'], preds], axis=1)
  return combined


In [ ]:
def backtest(data, model, predictors, start=1095, step=150): #here 1095 means using 3 year data to predict the 150 days of data
  all_predictions=[]

  for i in range(start, data.shape[0], step):
    train=data.iloc[0:i].copy()
    test=data.iloc[i:(i+step)].copy()
    predictions=predict(train, test, predictors, model)
    all_predictions.append(predictions)

  return pd.concat(all_predictions)


In [ ]:
from xgboost import XGBClassifier

model=XGBClassifier(random_state=1, learning_rate=.05, n_estimators=200)
predictions=backtest(btc, model, predictors)

In [ ]:
precision_score(predictions['target'], predictions['predictions'])

**We can see that our accuracy drops to 51%. lets try prediction with another set of predictiors.**

In [ ]:
def compute_rolling(btc):
    horizons = [2,7,60,365]
    new_predictors = ["close", "sentiment", "neg_sentiment"]

    for horizon in horizons:
        rolling_averages = btc.rolling(horizon, min_periods=1).mean()

        ratio_column = f"close_ratio_{horizon}"
        btc[ratio_column] = btc["close"] / rolling_averages["close"]

        edit_column = f"edit_{horizon}"
        btc[edit_column] = rolling_averages["edit_count"]

        rolling = btc.rolling(horizon, closed='left', min_periods=1).mean()
        trend_column = f"trend_{horizon}"
        btc[trend_column] = rolling["target"]

        new_predictors+= [ratio_column, trend_column, edit_column]
    return btc, new_predictors

In [ ]:
btc, new_predictors = compute_rolling(btc.copy())

In [ ]:
predictions = backtest(btc, model, new_predictors)

In [ ]:
precision_score(predictions["target"], predictions["predictions"])

In [ ]:
predictions

In [ ]:
btc.to_csv('BTC.csv')